In [ ]:
!pip install -q bert_score evaluate

In [ ]:
import pandas as pd
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import AutoPeftModelForCausalLM
from tqdm import tqdm

In [ ]:
test_data = df.sample(100, random_state=42)
pertanyaan_test = test_data['question'].tolist()

# Normalisasi referensi (huruf kecil semua)
jawaban_referensi = [str(ans).lower() for ans in test_data['answer'].tolist()]

Evaluasi Dialo-GPT

In [ ]:
tokenizer_dialo = AutoTokenizer.from_pretrained("/kaggle/input/models/muhammadzakikurnia/dialoo/pytorch/default/1/chatbot_fitness_final")
model_dialo = AutoModelForCausalLM.from_pretrained("/kaggle/input/models/muhammadzakikurnia/dialoo/pytorch/default/1/chatbot_fitness_final").to("cuda")

prediksi_dialo = []
for q in tqdm(pertanyaan_test, desc="DialoGPT Menjawab"):
    inputs = tokenizer_dialo(q + tokenizer_dialo.eos_token, return_tensors="pt").to("cuda")
    outputs = model_dialo.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer_dialo.eos_token_id)
    # Ambil teks jawaban saja (hilangkan pertanyaan)
    full_text = tokenizer_dialo.decode(outputs[0], skip_special_tokens=True)
    ans = full_text.replace(q, "").strip()
    prediksi_dialo.append(ans)

# Bersihkan VRAM GPU
del model_dialo
torch.cuda.empty_cache()


Evaluasi LoRA

In [ ]:
tokenizer_qwen = AutoTokenizer.from_pretrained("/kaggle/input/models/muhammadzakikurnia/gwen/pytorch/default/1/qwen_fitness_lora_final")
model_qwen = AutoPeftModelForCausalLM.from_pretrained(
    "/kaggle/input/models/muhammadzakikurnia/gwen/pytorch/default/1/qwen_fitness_lora_final",
    device_map={"": 0},
    torch_dtype=torch.float16
)

prediksi_qwen = []
for q in tqdm(pertanyaan_test, desc="Qwen Menjawab"):
    prompt = f"<|im_start|>system\nYou are a strict fitness dictionary. Answer directly without any pleasantries or conversational filler.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer_qwen(prompt, return_tensors="pt").to("cuda")
    outputs = model_qwen.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.01,
        top_p=1.0,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer_qwen.eos_token_id
    )
    full_text = tokenizer_qwen.decode(outputs[0], skip_special_tokens=True)
    ans = full_text.split("assistant\n")[-1].strip()

    # Normalisasi prediksi (huruf kecil semua)
    prediksi_qwen.append(ans.lower())

# Bersihkan VRAM
del model_qwen
torch.cuda.empty_cache()

Evaluasi GPT-Neo

In [ ]:
model_neo_id = "EleutherAI/gpt-neo-1.3B"
tokenizer_neo = AutoTokenizer.from_pretrained(model_neo_id)
# Set padding token
if tokenizer_neo.pad_token is None:
    tokenizer_neo.pad_token = tokenizer_neo.eos_token
model_neo = AutoModelForCausalLM.from_pretrained(model_neo_id, torch_dtype=torch.float16).to("cuda")

prediksi_neo = []
for q in tqdm(pertanyaan_test, desc="GPT-Neo Menjawab"):
    inputs = tokenizer_neo(q, return_tensors="pt").to("cuda")
    outputs = model_neo.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer_neo.pad_token_id)
    ans = tokenizer_neo.decode(outputs[0], skip_special_tokens=True).replace(q, "").strip()
    prediksi_neo.append(ans)

del model_neo
torch.cuda.empty_cache()

Evaluasi Llama

In [ ]:
model_llama_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_llama = AutoTokenizer.from_pretrained(model_llama_id)
model_llama = AutoModelForCausalLM.from_pretrained(model_llama_id, torch_dtype=torch.float16).to("cuda")

prediksi_llama = []
for q in tqdm(pertanyaan_test, desc="TinyLlama Menjawab"):
    # Menggunakan chat template khusus model Instruct/Chat
    messages = [{"role": "user", "content": q}]
    prompt = tokenizer_llama.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_llama(prompt, return_tensors="pt").to("cuda")

    outputs = model_llama.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer_llama.eos_token_id)
    # Memotong prompt agar hanya mengambil murni jawaban bot
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    ans = tokenizer_llama.decode(generated_ids, skip_special_tokens=True).strip()
    prediksi_llama.append(ans)

del model_llama
torch.cuda.empty_cache()